In [0]:


storage_account_name = "energysentineldl"
storage_account_key = "OiRgE+cLxjUGAE2GvfkNRuiIV04cs6kvxHzKI3nVcOU4xE4PpYrWImVDp7Szdh/JGJJUFXyBKCW5+AStLuP97Q=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/")
print(" Connected!")


✅ Connected!


In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

STORAGE_ACCOUNT = "energysentineldl"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/sensor_readings"
GOLD_PLANT_SUMMARY_PATH = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/plant_summary"
GOLD_SENSOR_SUMMARY_PATH = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/sensor_summary"
GOLD_HOURLY_TRENDS_PATH = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/hourly_trends"

# قراءة Silver
df_silver = spark.read.format("delta").load(SILVER_PATH)

df_silver = df_silver.withColumn("date", F.to_date("timestamp")) \
                      .withColumn("hour", F.hour("timestamp"))

# ---------------------------------------------------
# 1) Plant Summary 

df_plant_summary = (
    df_silver.groupBy("plant_id")
    .agg(
        F.count("*").alias("total_readings"),
        F.avg("power_consumption").alias("avg_power_consumption"),
        F.avg("voltage").alias("avg_voltage"),
        F.avg("temperature").alias("avg_temperature"),
        F.avg("frequency").alias("avg_frequency"),
        F.sum("is_anomaly").alias("total_anomalies"),
    )
    .withColumn(
        "anomaly_rate_pct",
        F.round((F.col("total_anomalies") / F.col("total_readings")) * 100, 2)
    )
)

display(df_plant_summary)

# ---------------------------------------------------
# 2) Sensor Summary: إحصائيات لكل sensor لوحده
df_sensor_summary = (
    df_silver.groupBy("sensor_id", "plant_id")
    .agg(
        F.count("*").alias("total_readings"),
        F.avg("power_consumption").alias("avg_power_consumption"),
        F.max("power_consumption").alias("max_power_consumption"),
        F.avg("temperature").alias("avg_temperature"),
        F.max("temperature").alias("max_temperature"),
        F.sum("is_anomaly").alias("total_anomalies"),
    )
    .withColumn(
        "anomaly_rate_pct",
        F.round((F.col("total_anomalies") / F.col("total_readings")) * 100, 2)
    )
    .orderBy(F.desc("anomaly_rate_pct"))
)

display(df_sensor_summary)

# ---------------------------------------------------
# 3) Hourly Trends: اتجاه الاستهلاك والأعطال بالساعة (لكل يوم/plant)
df_hourly_trends = (
    df_silver.groupBy("plant_id", "date", "hour")
    .agg(
        F.avg("power_consumption").alias("avg_power_consumption"),
        F.avg("temperature").alias("avg_temperature"),
        F.sum("is_anomaly").alias("anomalies_count"),
        F.count("*").alias("readings_count"),
    )
    .orderBy("plant_id", "date", "hour")
)

display(df_hourly_trends)

# ---------------------------------------------------
# الكتابة إلى Gold
df_plant_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PLANT_SUMMARY_PATH)
df_sensor_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_SENSOR_SUMMARY_PATH)
df_hourly_trends.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("plant_id").save(GOLD_HOURLY_TRENDS_PATH)

print("تم حفظ الـ 3 جداول في Gold ")

plant_id,total_readings,avg_power_consumption,avg_voltage,avg_temperature,avg_frequency,total_anomalies,anomaly_rate_pct
PLANT_A,16925,483.307924963073,219.0893760709012,67.0832649926143,50.040707237814054,809,4.78
PLANT_B,10155,245.13464500246113,219.1210497291977,58.268796651896,50.04581486952233,517,5.09
PLANT_C,6770,112.71966174298389,219.16154505170044,47.485747415066655,50.059539143279196,361,5.33


sensor_id,plant_id,total_readings,avg_power_consumption,max_power_consumption,avg_temperature,max_temperature,total_anomalies,anomaly_rate_pct
SENSOR_C01,PLANT_C,3385,117.69919645494845,288.38,48.34871491875921,97.13,188,5.55
SENSOR_B03,PLANT_B,3385,264.7334564254061,663.09,60.275025110782806,120.19,173,5.11
SENSOR_C02,PLANT_C,3385,107.74012703101918,278.38,46.62277991137367,95.33,173,5.11
SENSOR_B01,PLANT_B,3385,244.93781683899647,622.18,58.34737370753336,119.11,173,5.11
SENSOR_A02,PLANT_A,3385,470.4261093057594,1175.8,65.93662924667649,135.62,172,5.08
SENSOR_B02,PLANT_B,3385,225.73266174298467,550.87,56.183991137370555,112.08,171,5.05
SENSOR_A01,PLANT_A,3385,489.9361683899556,1210.96,67.97769276218612,137.83,169,4.99
SENSOR_A04,PLANT_A,3385,450.5640679468233,1108.48,63.0012259970458,124.33,158,4.67
SENSOR_A03,PLANT_A,3385,507.56593796159655,1247.03,69.7081565731166,138.62,158,4.67
SENSOR_A05,PLANT_A,3385,498.0473412112261,1217.41,68.79262038404738,138.94,152,4.49


plant_id,date,hour,avg_power_consumption,avg_temperature,anomalies_count,readings_count
PLANT_A,2026-06-28,22,478.2586795327572,66.3711416962927,484,9845
PLANT_A,2026-06-28,23,490.32908615819207,68.07349858757061,325,7080
PLANT_B,2026-06-28,22,242.01441171491578,57.65936177416615,278,5907
PLANT_B,2026-06-28,23,249.47344397363392,59.11623822975525,239,4248
PLANT_C,2026-06-28,22,111.46163026917216,46.959327069578414,196,3938
PLANT_C,2026-06-28,23,114.4690007062147,48.2177542372882,165,2832


✅ تم حفظ الـ 3 جداول في Gold بنجاح
